In [0]:
# ============================================================
# PAYCE DATA PLATFORM — Silver Layer Configuration
# Notebook:    02_silver_processing
# Layer:       Silver
# Description: Reusable config — run this cell first each session
# ============================================================

# S3 base paths
S3_RAW_BASE    = "s3://payce-data-platform/raw"
S3_BRONZE_BASE = "s3://payce-data-platform/bronze"
S3_SILVER_BASE = "s3://payce-data-platform/silver"

# Catalog preferences
CATALOG = "payce"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

print("Payce Silver config loaded")
print(f" CATALOG: {CATALOG}")
print(f" BRONZE:  {BRONZE}")
print(f" SILVER:  {SILVER}")

In [0]:
# Check existing catalogs
spark.sql("SHOW CATALOGS").show()

In [0]:
# CREATE THE PAYCE CATALOG
spark.sql("CREATE CATALOG IF NOT EXISTS payce")

# CREATE the medallion schema
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS payce.gold")

# CONFIRM
spark.sql("SHOW SCHEMAS IN payce").show()

In [0]:
# ============================================================
# Register existing Bronze Delta files as Unity Catalog tables
# ============================================================

S3_BRONZE_BASE = "s3://payce-data-platform/bronze"

# List of datasets to register
datasets = ["paysim", "ieee_cis", "home_credit", "lending_club"]

for ds in datasets:
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS payce.bronze.{ds}
              USING DELTA
              LOCATION '{S3_BRONZE_BASE}/{ds}/'
              """)
    print(f" Registed payce.bronze.{ds}")

# Confirm all tables
spark.sql("SHOW TABLES IN payce.bronze").show()

In [0]:
# Test querying via catalog
spark.sql("SELECT * from payce.bronze.paysim LIMIT 5").show()

In [0]:
# ===============================================================
# SILVER - PaySim Exploration
# Understand the data before transforming
# ===============================================================

# Load Bronze PaySim
df = spark.table(f"{BRONZE}.paysim")

# 1. Look at the schema (column names + data types)
print("SCHEMA:")
df.printSchema()

# 2. Row Count
print(f"\n Total rows: {df.count():,}")

# 3. Preview a few rows
df.show(5)

In [0]:
# Check transaction types
print("TRANSACTION TYPES:")
df.groupBy("type").count().show()

# Check if _rescued_data has any bad records
print("RESCUED (bad) records:")
df.filter("_rescued_data IS NOT NULL").count()

In [0]:
from pyspark.sql.functions import col, when

# Load Bronze
df = spark.table(f"{BRONZE}.paysim")

# Cast columns to correct data types
df_silver = (
    df
    .withColumn("step",             col("step").cast("int"))
    .withColumn("amount",           col("amount").cast("double"))
    .withColumn("oldbalanceOrg",    col("oldbalanceOrg").cast("double"))
    .withColumn("newbalanceOrig",   col("newbalanceOrig").cast("double"))
    .withColumn("oldbalanceDest",   col("oldbalanceDest").cast("double"))
    .withColumn("newbalanceDest",   col("newbalanceDest").cast("double"))
    .withColumn("isFraud",          col("isFraud").cast("int"))
    .withColumn("isFlaggedFraud",   col("isFlaggedFraud").cast("int"))
)

# Drop the _rescued_data column (confirmed as empty)
df_silver = df_silver.drop("_rescued_data")


# Remove the duplicates rows
df_silver = df_silver.dropDuplicates()

# Add a derived column - balance change for the origin account 
df_silver = df_silver.withColumn(
    "balance_change_orig",      col("oldbalanceOrg") - col("newbalanceOrig")
)


# Write to silver as a managed delta table 

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.paysim")
)

print("payce.silver.paysim created successfully")

In [0]:
%sql
select count(*) from payce.silver.paysim

In [0]:
df_silver.printSchema()

In [0]:
# ============================================================
# SILVER — IEEE-CIS Exploration
# ============================================================

# Load Bronze IEEE-CIS
df_ieee = spark.table(f"{BRONZE}.ieee_cis")

# 1. Schema
print("SCHEMA: ")
df_ieee.printSchema()

# 2. Row Count
print("f\n Total rows: {df_ieee.count():,}")

# 3. How many columns
print(f"Total Columns: {len(df_ieee.columns)}")

In [0]:
# Check Fraud Distribution 
df_ieee.groupBy("isFraud").count().show()

In [0]:
### isFraud contains indicators 1,0, NULL, 0.5. We need to remove and exclude NULLS and 0.5 from this data

In [0]:
from pyspark.sql.functions import col 

# Keep only labelled training rows
df_ieee_clean = df_ieee.filter(col("isFraud").isin("0","1"))

# Confirm the fix
df_ieee_clean.groupBy("isFraud").count().show()

In [0]:
# ============================================================
# SILVER — IEEE-CIS Transformation
# Source: payce.bronze.ieee_cis
# Target: payce.silver.ieee_cis
# Approach: Filter to labelled rows, curate ~15 BNPL-relevant
#           columns from the original 473
# ============================================================

from pyspark.sql.functions import col

# Load Bronze 
df_ieee = spark.table(f"{BRONZE}.ieee_cis")

# Keep only real and fraud indicators
df_ieee = df_ieee.filter(col("isFraud").isin("0","1"))

# Select only BNPL relevant columns from 473 
selected_columns = [
    "TransactionID",        # unique transaction key
    "isFraud",              # fraud income
    "TransactionDT",        # time (seconds from reference)
    "TransactionAmt",       # transaction amount in USD
    "ProductCD",            # product code
    "card4",                # card network
    "card6",                # card type (credit/debit)
    "P_emaildomain",        # purchaser email domain
    "addr1",                # billing region
    "C1",                   # count signal
    "C2",                   # count signal
    "D1",                   # days since first transaction
    "M1",                   # match flag
    "M6",                   # match flag
]

df_ieee = df_ieee.select(*selected_columns)


# Cast columns to correct data types
df_ieee_silver = (
    df_ieee
    .withColumn("isFraud",                   col("isFraud").cast("int"))
    .withColumn("TransactionDT",             col("TransactionDT").cast("long"))
    .withColumn("TransactionAmt",            col("TransactionAmt").cast("double"))
    .withColumn("addr1",                     col("addr1").cast("double").cast("int"))
    .withColumn("C1",                        col("C1").cast("double"))
    .withColumn("C2",                        col("C2").cast("double"))
    .withColumn("D1",                        col("D1").cast("double"))
)

# Remove duplicates
df_ieee_silver = df_ieee_silver.dropDuplicates(["TransactionID"])

# Write Silver 
(
    df_ieee_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.ieee_cis")
)

print("payce.silver.ieee_cis created successfully")

In [0]:
%sql
select count(*) from payce.silver.ieee_cis 

In [0]:
# ============================================================
# SILVER — Home Credit Exploration
# ============================================================

df_hc = spark.table(f"{BRONZE}.home_credit")

# 1. How many columns?
print(f"Total columns: {len(df_hc.columns)}")

# 2. Print total rows
print(f"Total rows: {df_hc.count():,}")

# 3. Look at first 30 columns names to see the patchwork
print("\n First 30 Columns")
print(df_hc.columns[:30])

In [0]:
# ============================================================
# SILVER — Home Credit: read application_train directly from raw
# ============================================================

# Read just the main application file from S3
df_app = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("s3://payce-data-platform/raw/home_credit/application_train.csv")
)

# Confirm its clean
print(f"Rows: {df_app.count():,}")
print(f"Columns: {len(df_app.columns)}")

# Check the Target distribution
df_app.groupBy("TARGET").count().show()

In [0]:
# ============================================================
# SILVER — Home Credit Transformation
# Source: raw application_train.csv
# Target: payce.silver.home_credit
# Approach: Curate ~12 BNPL-relevant columns from 122
# ============================================================

from pyspark.sql.functions import col, round as spark_round

# Read the clean application file from raw
df_app = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("s3://payce-data-platform/raw/home_credit/application_train.csv")
)

# Select only BNPL relevant columns
selected_columns = [
    "SK_ID_CURR",            # customer ID
    "TARGET",                # defaulted (1) or repaid (0)
    "NAME_CONTRACT_TYPE",    # loan type
    "AMT_INCOME_TOTAL",      # income
    "AMT_CREDIT",            # loan amount
    "AMT_ANNUITY",           # repayment per period
    "AMT_GOODS_PRICE",       # price of goods
    "NAME_INCOME_TYPE",      # employment type
    "NAME_EDUCATION_TYPE",   # education level
    "DAYS_BIRTH",            # age in days (negative)
    "DAYS_EMPLOYED",         # employment length in days (negative)
    "CNT_CHILDREN"           # number of children
]

df_hc = df_app.select(*selected_columns)


# Cast columns 
df_hc_silver = (
    df_hc
    .withColumn("SK_ID_CURR",       col("SK_ID_CURR").cast("int"))
    .withColumn("TARGET",           col("TARGET").cast("int"))
    .withColumn("AMT_INCOME_TOTAL", col("AMT_INCOME_TOTAL").cast("double"))
    .withColumn("AMT_CREDIT",       col("AMT_CREDIT").cast("double"))
    .withColumn("AMT_ANNUITY",      col("AMT_ANNUITY").cast("double"))
    .withColumn("AMT_GOODS_PRICE",  col("AMT_GOODS_PRICE").cast("double"))
    .withColumn("DAYS_BIRTH",       col("DAYS_BIRTH").cast("int"))
    .withColumn("DAYS_EMPLOYED",    col("DAYS_EMPLOYED").cast("int"))
    .withColumn("CNT_CHILDREN",     col("CNT_CHILDREN").cast("int"))
)


# ③ Add reusable derived fact — convert negative DAYS_BIRTH to age in years
df_hc_silver = df_hc_silver.withColumn(
    "age_years",
    spark_round(col("DAYS_BIRTH") / -365.25, 0).cast("int")
)


# Remove duplicates
df_hc_silver = df_hc_silver.dropDuplicates(["SK_ID_CURR"])


# Write Silver
(
df_hc_silver.write
.format("delta")
.mode("overwrite")
.saveAsTable(f"{SILVER}.home_credit")
)

print("payce.silver.home_credit created successfully")

In [0]:
%sql
select * from payce.silver.home_credit limit 5

In [0]:
# ============================================================
# SILVER — LendingClub Exploration
# ============================================================

df_lc = spark.table(f"{BRONZE}.lending_club")

# 1. columns count
print(f"Total Columns: {len(df_lc.columns)}")

# 2. Row Count
print(f"Total Row count: {df_lc.count():,}")

In [0]:
# Check what raw LendingClub files exist
files = dbutils.fs.ls("s3://payce-data-platform/raw/lending_club/")
for f in files:
    print(f.name, f"({f.size / (1024*1024):.0f} MB)")

In [0]:
# ============================================================
# SILVER — LendingClub: read accepted loans directly from raw
# ============================================================

# Read the accepted loans file (Spark handles .gz automatically)
df_accepted = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("s3://payce-data-platform/raw/lending_club/accepted_2007_to_2018Q4.csv.gz")
)

# Confirm it's clean
print(f"Rows: {df_accepted.count():,}")
print(f"Columns: {len(df_accepted.columns)}")

# Check loan status distribution (the key performance column)
df_accepted.groupBy("loan_status").count().show(truncate=False)

In [0]:
# ============================================================
# SILVER — LendingClub Transformation
# Source: raw accepted_2007_to_2018Q4.csv.gz
# Target: payce.silver.lending_club
# Approach: Curate ~10 BNPL columns from 151, simplify loan_status
# ============================================================

from pyspark.sql.functions import col, when

# Read accepted loans from raw
df_accepted = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("s3://payce-data-platform/raw/lending_club/accepted_2007_to_2018Q4.csv.gz")
)


# Selected BNPL relevant columns
selected_columns = [
    "id",              # loan ID
    "loan_amnt",       # loan amount
    "term",            # loan term (36/60 months)
    "int_rate",        # interest rate
    "grade",           # LendingClub risk grade (A-G)
    "annual_inc",      # borrower annual income
    "purpose",         # reason for loan
    "dti",             # debt-to-income ratio
    "loan_status"      # repayment outcome
]

df_lc = df_accepted.select(*selected_columns)

In [0]:
df_lc.select("loan_status").distinct().show()

In [0]:
# Drop junk rows (NULL and malformed statuses)
valid_statuses = [
     "Fully Paid", "Current", "Charged Off", "Default",
    "Late (31-120 days)", "Late (16-30 days)", "In Grace Period",
    "Does not meet the credit policy. Status:Charged Off",
    "Does not meet the credit policy. Status:Fully Paid"
]

df_lc = df_lc.filter(col("loan_status").isin(valid_statuses))

In [0]:
from pyspark.sql.functions import expr

# Cast columns to correct data types (use try_cast to handle malformed values)
df_lc_silver = (
    df_lc
    .withColumn("annual_inc",   expr("try_cast(annual_inc as double)"))
    .withColumn("dti",          expr("try_cast(dti as double)"))
    )

# Add simplified loan outcomes flag (reusable business flag)
df_lc_silver = df_lc_silver.withColumn("loan_outcome",
                                when(col("loan_status").isin(
                                    "Fully Paid", "Current", "In Grace Period",
                                    "Does not meet the credit policy. Status:Fully Paid"
                                ), "Performing")
                                .otherwise("Defaulted/At Risk")
)


# Remove duplicates
df_lc_silver = df_lc_silver.dropDuplicates(["id"])


# Write to silver
(
df_lc_silver.write
.format("delta")
.mode("overwrite")
.saveAsTable(f"{SILVER}.lending_club")
)

print("payce silver lending club created successfully")

In [0]:
# Verify Silver LendingClub 
df_lc_silver = spark.table(f"{SILVER}.lending_club")

print("SILVER SCHEMA:", df_lc_silver.printSchema())

In [0]:
print(f"\n Total Rows:, {df_lc_silver.count():,}")

# check the loan outcome split
df_lc_silver.groupBy("loan_outcome").count().show()